## 03 Classification Homework/Project

In [82]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.preprocessing import OneHotEncoder

### Download data and read it

In [4]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv'

In [5]:
!wget $data

--2026-07-22 01:54:06--  https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 80876 (79K) [text/plain]
Saving to: ‘course_lead_scoring.csv’

course_lead_scoring 100%[===================>]  78.98K  --.-KB/s    in 0.002s  

2026-07-22 01:54:06 (31.4 MB/s) - ‘course_lead_scoring.csv’ saved [80876/80876]



In [39]:
df = pd.read_csv(data)

In [40]:
df.shape

(1462, 9)

In [41]:
df.head()

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1


In [42]:
df.dtypes

lead_source                     str
industry                        str
number_of_courses_viewed      int64
annual_income               float64
employment_status               str
location                        str
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

### Data Preparation

In [43]:
X = df.drop(columns = 'converted')

In [57]:
y = df.converted

In [44]:
#check missing values for each features
X.isnull().sum()

lead_source                 128
industry                    134
number_of_courses_viewed      0
annual_income               181
employment_status           100
location                     63
interaction_count             0
lead_score                    0
dtype: int64

In [46]:
categorical_cols = X.dtypes.index[X.dtypes =='str']

In [47]:
categorical_cols

Index(['lead_source', 'industry', 'employment_status', 'location'], dtype='str')

In [48]:
numerical_cols = X.columns.difference(categorical_cols)

In [49]:
numerical_cols

Index(['annual_income', 'interaction_count', 'lead_score',
       'number_of_courses_viewed'],
      dtype='str')

In [50]:
X[categorical_cols] = X[categorical_cols].fillna('NA')

In [51]:
X[numerical_cols] = X[numerical_cols].fillna(0)

In [52]:
X.isnull().sum()

lead_source                 0
industry                    0
number_of_courses_viewed    0
annual_income               0
employment_status           0
location                    0
interaction_count           0
lead_score                  0
dtype: int64

In [53]:
X.dtypes

lead_source                     str
industry                        str
number_of_courses_viewed      int64
annual_income               float64
employment_status               str
location                        str
interaction_count             int64
lead_score                  float64
dtype: object

In [54]:
X.industry.value_counts()

industry
retail           203
finance          200
other            198
healthcare       187
education        187
technology       179
manufacturing    174
NA               134
Name: count, dtype: int64

### Question 1: retail

In [55]:
X[numerical_cols].corr()

,annual_income,interaction_count,lead_score,number_of_courses_viewed
annual_income,1.000000,0.027036,0.015610,0.009770
interaction_count,0.027036,1.000000,0.009888,-0.023565
lead_score,0.015610,0.009888,1.000000,-0.004879
number_of_courses_viewed,0.009770,-0.023565,-0.004879,1.000000


### Question 2: 0.027036 interaction_count  & annual_income

In [58]:
# Step 1: Split into Train+Val (80%) and Test (20%)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Step 2: Split Train+Val into final Train (75% of 80% = 60%) and Val (25% of 80% = 20%)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

In [59]:
X_train.shape,  X_val.shape, X_test.shape

((876, 8), (293, 8), (293, 8))

In [70]:
X.columns

Index(['lead_source', 'industry', 'number_of_courses_viewed', 'annual_income',
       'employment_status', 'location', 'interaction_count', 'lead_score'],
      dtype='str')

In [72]:
X.dtypes

lead_source                     str
industry                        str
number_of_courses_viewed      int64
annual_income               float64
employment_status               str
location                        str
interaction_count             int64
lead_score                  float64
dtype: object

In [81]:
for col in categorical_cols:
    mi = mutual_info_score(X_train[col], y_train)
    print (f'mi score between {col} and response variable is {mi}')

mi score between lead_source and response variable is 0.027718699139711492
mi score between industry and response variable is 0.005958887591320947
mi score between employment_status and response variable is 0.007695750218801714
mi score between location and response variable is 0.0014310098448050215


###  Question 3: lead_scource

In [91]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')

In [92]:
encoded_categorical_cols = encoder.fit_transform(X_train[categorical_cols])

In [93]:
display(encoded_categorical_cols)

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 1., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(876, 23))